# PS2 overlap diagnostic

Read-only. Does not write to S3, does not touch the Gold layer, does not publish, and does
not modify the PS2 notebook or any PS2 output.

**Why this exists.** The episode-closure diagnostic disproved the theory that OOS episodes
never close: `clear_per_set` was ~1.00 on every fleet, `pct_unclosed` was 0.00, and
`device_outage` had zero rows with a null `outage_end`. Boundary censoring at 11 April is
negligible. But the recorded totals still imply **22.9 out-of-service hours per validator
per day**, sustained over 210 days, and a 24-hour day cannot contain that.

One candidate remains. PS2 computes `hardware_oos_minutes` as `sum(device_outage.duration_min)`.
If a device can be in several OOS episodes at once -- different components failing together,
which is precisely what PS2's cascade thesis claims -- then that sum counts the same
wall-clock minute once per concurrent episode. This notebook measures that rather than
assuming it.

**What it does.**

- **A** Does `duration_min` even agree with `outage_end - outage_start`? Plus the 7-day
  duration cap and exact-duplicate intervals.
- **B** The decisive test: merge overlapping intervals per device and compare the union
  against the sum. `overlap_factor` is how many times the average out-of-service minute is
  being counted.
- **C** How deep concurrency goes, and what *kind* it is -- different subsystems (legitimate
  multi-component recording, fixable in the notebook) or repeated event ids (a source
  defect, which is a conversation with Cubic instead).
- **E** The restated share of available device-hours, computed from the union and clipped
  per calendar day so no device can exceed 1440 minutes in a day by construction. The
  ceiling check must print 0.

**Verification.** The interval-merge and sweep-line logic were checked against a
brute-force minute-set union over 300 randomised cases plus the awkward shapes real data
contains -- duplicate, nested, touching and zero-length intervals. Zero mismatches. On the
edge-case fixture the union is 30 minutes where the naive sum is 45, which is exactly the
double-counting this run is built to expose.

**To run.** Set `PS2_NOTEBOOK_PATH` if your repo clone is not at
`~/Chicago-Ventra-Mars-Cubic-Analysis`, then Run All. Only PS2 cells 0-4 are executed.
Cell 2 is the heavy one; cells 2 and 3 cache their intermediates so cell 3 does not
recompute the merge.

**What it still does not test.** Whether the underlying episodes are real at all. A union
of spurious intervals is still spurious.


In [ ]:
# =============================================================================
# PS2 OVERLAP DIAGNOSTIC -- CELL 1 of 3: borrow the PS2 setup, change nothing
#
# READ-ONLY. Never writes to S3, never touches Gold, never publishes, never
# modifies the PS2 notebook. Same discipline as the episode-closure diagnostic:
# it execs PS2 code cells 0..4, which are pure setup, so the source paths, the
# fleet mapping and the column contracts cannot drift from the pipeline being
# explained. PS2 cell 5 onward is deliberately NOT run.
#
# WHY THIS RUN EXISTS. The closure diagnostic disproved the unclosed-episode
# theory: clear_per_set was ~1.00 on every fleet, pct_unclosed was 0.00, and
# device_outage had zero null_end rows. But the recorded totals still imply
# 22.9 out-of-service hours per validator per day, which a 24-hour day cannot
# contain. The remaining candidate is that PS2 sums duration_min across
# episodes that overlap in time on the same device, so the same wall-clock
# minute is counted once per concurrent episode. This notebook measures that
# directly instead of assuming it.
# =============================================================================
import os, json
from pathlib import Path

os.environ['PS2_RUN_MODE'] = 'REPLAY'
os.environ['PS2_EXPORT_PREFIX'] = 'ps2_diagnostic_overlap_outputs'
os.environ.setdefault('PS2_MAX_SOURCE_LAG_DAYS', '150')

PS2_NOTEBOOK = Path(os.getenv(
    'PS2_NOTEBOOK_PATH',
    Path.home() / 'Chicago-Ventra-Mars-Cubic-Analysis' / 'notebooks'
    / 'ps2_cascading_failure' / 'PS2_Failure_Patterns_v2_5_3_Storage_Safe.ipynb',
))
if not PS2_NOTEBOOK.exists():
    raise FileNotFoundError(
        f'Cannot find the PS2 notebook at {PS2_NOTEBOOK}. Set PS2_NOTEBOOK_PATH to its '
        'location in the synced repo clone.'
    )

_nb = json.loads(PS2_NOTEBOOK.read_text(encoding='utf-8'))
_code = [''.join(c['source']) for c in _nb['cells'] if c['cell_type'] == 'code']
SETUP_CELLS = 5
if len(_code) < SETUP_CELLS:
    raise RuntimeError(f'Expected at least {SETUP_CELLS} code cells in PS2, found {len(_code)}.')

_forbidden = ('put_object', 'upload_file', 'saveAsTable', '.write.', 'delete_object')
for _i in range(SETUP_CELLS):
    _hit = [t for t in _forbidden if t in _code[_i]]
    if _hit:
        raise RuntimeError(
            f'PS2 setup cell {_i} now contains a write operation {_hit}; this diagnostic '
            'will not exec it. Review the notebook before continuing.'
        )

for _i in range(SETUP_CELLS):
    print(f'--- exec PS2 code cell {_i} ({len(_code[_i]):,} chars)')
    exec(compile(_code[_i], f'<PS2 code cell {_i}>', 'exec'), globals())

for _needed in ('spark', 'read_source', 'canonical_device_category', 'truthy',
                'TARGET_DEVICE_CATEGORIES', 'ANALYSIS_LOOKBACK_DAYS', 'SOURCE'):
    if _needed not in globals():
        raise RuntimeError(f'PS2 setup did not provide {_needed!r}; cannot continue.')

print()
print('Setup borrowed from PS2. No PS2 load or publish cell was run.')
print('outage ->', SOURCE['outage']['path'])


In [ ]:
# =============================================================================
# PS2 OVERLAP DIAGNOSTIC -- CELL 2 of 3: do episodes overlap, and by how much?
#
# PS2 computes hardware_oos_minutes as sum(device_outage.duration_min). If a
# device can be in several OOS episodes at once -- different components failing
# together, which is what PS2's cascade thesis says happens -- then that sum
# counts the same wall-clock minute once per concurrent episode.
#
# The honest quantity is the UNION of the intervals per device. This cell
# computes both and compares them. It also checks four things that would each
# change the interpretation:
#
#   A  does duration_min agree with (outage_end - outage_start)?
#   B  merged union minutes vs summed minutes, per fleet
#   C  how deep does concurrency actually go, and what distinguishes the
#      simultaneous episodes from one another?
#   D  exact-duplicate intervals, and the 7-day duration cap
#
# READ-ONLY.
# =============================================================================
import datetime as dt
import pandas as pd
from pyspark.sql import functions as F, Window
from pyspark import StorageLevel

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)

TARGETS = list(TARGET_DEVICE_CATEGORIES)
DAY = 1440.0

# The closure diagnostic established the window from the event table. Rebuild it
# the same way so the two runs are directly comparable.
_ev = (
    read_source('label_events',
                ['DEVICE_ID', 'EVENT_DTM', 'EVENT_STATE_TYPE_NAME', 'is_hardware_oos_event',
                 'mars_device_category'], [])
    .withColumn('fleet', canonical_device_category(F.col('mars_device_category')))
    .where(F.col('fleet').isin(*TARGETS))
    .where(truthy(F.col('is_hardware_oos_event')))
)
WIN_END = _ev.agg(F.max(F.col('EVENT_DTM').cast('timestamp')).alias('m')).first()['m']
WIN_START = WIN_END - dt.timedelta(days=ANALYSIS_LOOKBACK_DAYS)
WIN_DAYS = ANALYSIS_LOOKBACK_DAYS
print(f'Window: {WIN_START} -> {WIN_END}  ({WIN_DAYS} days)')
print()

og = read_source(
    'outage',
    ['DEVICE_ID', 'outage_start', 'outage_end', 'duration_min', 'mars_device_category'],
    ['source_event_id', 'is_chargeable', 'failure_level', 'component_subsystem',
     'COMPONENT_SERIAL_NBR'],
)
print('columns projected from device_outage:', og.columns)
print()

og = (
    og
    .withColumn('fleet', canonical_device_category(F.col('mars_device_category')))
    .withColumn('device_id', F.col('DEVICE_ID').cast('string'))
    .withColumn('start_ts', F.col('outage_start').cast('timestamp'))
    .withColumn('end_ts', F.col('outage_end').cast('timestamp'))
    .withColumn('dur_min', F.col('duration_min').cast('double'))
    .where(F.col('fleet').isin(*TARGETS))
    .where((F.col('start_ts') >= F.lit(WIN_START)) & (F.col('start_ts') <= F.lit(WIN_END)))
    .where(F.col('device_id').isNotNull() & F.col('start_ts').isNotNull() & F.col('end_ts').isNotNull())
    .withColumn('span_min', (F.col('end_ts').cast('long') - F.col('start_ts').cast('long')) / 60.0)
    .persist(StorageLevel.DISK_ONLY)
)
n_rows = og.count()
# The closure diagnostic found zero null outage_end rows, but never drop rows
# without saying how many. If this is not 0, the union below is incomplete.
_all_in_window = (
    read_source('outage', ['DEVICE_ID', 'outage_start', 'outage_end', 'duration_min', 'mars_device_category'], [])
    .withColumn('fleet', canonical_device_category(F.col('mars_device_category')))
    .where(F.col('fleet').isin(*TARGETS))
    .where((F.col('outage_start').cast('timestamp') >= F.lit(WIN_START))
           & (F.col('outage_start').cast('timestamp') <= F.lit(WIN_END)))
    .count()
)
print(f'device_outage rows in window: {n_rows:,}')
print(f'rows excluded for a null device_id, start or end: {_all_in_window - n_rows:,} '
      f'(must be 0 for the union to be complete)')
print()

# ---------------------------------------------------------------- A: duration vs span
# If duration_min and (end - start) disagree, merging on the interval and
# summing duration_min are measuring two different things, and that has to be
# known before anything else is interpreted.
print('A.1  Does duration_min agree with (outage_end - outage_start)?')
agree = (
    og.groupBy('fleet').agg(
        F.count('*').alias('episodes'),
        F.sum(F.when(F.abs(F.col('dur_min') - F.col('span_min')) <= 0.5, 1).otherwise(0)).alias('agree_within_30s'),
        F.sum(F.when(F.col('dur_min') > F.col('span_min') + 0.5, 1).otherwise(0)).alias('duration_gt_span'),
        F.sum(F.when(F.col('dur_min') < F.col('span_min') - 0.5, 1).otherwise(0)).alias('duration_lt_span'),
        F.sum(F.when(F.col('end_ts') < F.col('start_ts'), 1).otherwise(0)).alias('end_before_start'),
        F.expr('percentile_approx(span_min - dur_min, 0.5)').alias('p50_span_minus_duration'),
        F.max(F.col('span_min') - F.col('dur_min')).alias('max_span_minus_duration'),
        F.sum('dur_min').alias('sum_duration_min'),
        F.sum('span_min').alias('sum_span_min'),
    ).toPandas()
)
agree['pct_agree'] = (100.0 * agree['agree_within_30s'] / agree['episodes']).round(2)
print(agree.sort_values('fleet').to_string(index=False))
print()

# ---------------------------------------------------------------- D: cap and duplicates
print('A.2  The 7-day cap, and exact-duplicate intervals')
dup_key = ['fleet', 'device_id', 'start_ts', 'end_ts']
dup = (
    og.groupBy(*dup_key).agg(F.count('*').alias('n'))
    .groupBy('fleet').agg(
        F.count('*').alias('distinct_intervals'),
        F.sum('n').alias('rows'),
        F.sum(F.when(F.col('n') > 1, F.col('n') - 1).otherwise(0)).alias('duplicate_rows'),
        F.max('n').alias('max_rows_on_one_interval'),
    ).toPandas()
)
cap = (
    og.groupBy('fleet').agg(
        F.sum(F.when(F.col('dur_min') >= 10080.0, 1).otherwise(0)).alias('at_7_day_cap'),
        F.sum(F.when(F.col('span_min') > 10080.0, 1).otherwise(0)).alias('span_over_7_days'),
        F.sum(F.when(F.col('dur_min') == 0.0, 1).otherwise(0)).alias('zero_duration'),
    ).toPandas()
)
capdup = dup.merge(cap, on='fleet')
capdup['pct_duplicate_rows'] = (100.0 * capdup['duplicate_rows'] / capdup['rows']).round(2)
print(capdup.sort_values('fleet').to_string(index=False))
print()

# ---------------------------------------------------------------- B: the union
# Merge overlapping and touching intervals per device. This is the standard
# sweep: a new merged block starts only when an interval begins strictly after
# the furthest end seen so far on that device.
print('B.1  Merging overlapping intervals per device (this is the decisive test)')
w_dev = Window.partitionBy('fleet', 'device_id').orderBy('start_ts', 'end_ts')
merged = (
    og.select('fleet', 'device_id', 'start_ts', 'end_ts', 'dur_min')
    .withColumn('_prior_max_end', F.max('end_ts').over(w_dev.rowsBetween(Window.unboundedPreceding, -1)))
    .withColumn('_is_new_block',
                F.when(F.col('_prior_max_end').isNull(), F.lit(1))
                 .when(F.col('start_ts') > F.col('_prior_max_end'), F.lit(1))
                 .otherwise(F.lit(0)))
    .withColumn('_block', F.sum('_is_new_block').over(w_dev.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
    .groupBy('fleet', 'device_id', '_block')
    .agg(F.min('start_ts').alias('block_start'),
         F.max('end_ts').alias('block_end'),
         F.count('*').alias('episodes_in_block'),
         F.sum('dur_min').alias('summed_duration_min'))
    .withColumn('block_minutes', (F.col('block_end').cast('long') - F.col('block_start').cast('long')) / 60.0)
    .persist(StorageLevel.DISK_ONLY)
)
n_blocks = merged.count()
print(f'merged downtime blocks: {n_blocks:,} (from {n_rows:,} episodes)')
print()

union_by_fleet = (
    merged.groupBy('fleet').agg(
        F.countDistinct('device_id').alias('devices'),
        F.count('*').alias('merged_blocks'),
        F.sum('block_minutes').alias('union_minutes'),
        F.sum('summed_duration_min').alias('summed_minutes'),
        F.max('episodes_in_block').alias('max_episodes_in_one_block'),
        F.expr('percentile_approx(episodes_in_block, 0.5)').alias('p50_episodes_per_block'),
        F.expr('percentile_approx(episodes_in_block, 0.99)').alias('p99_episodes_per_block'),
    ).toPandas()
)
union_by_fleet['overlap_factor'] = (union_by_fleet['summed_minutes'] / union_by_fleet['union_minutes']).round(3)
union_by_fleet['minutes_double_counted'] = union_by_fleet['summed_minutes'] - union_by_fleet['union_minutes']
union_by_fleet['pct_of_sum_that_is_double_counted'] = (
    100.0 * union_by_fleet['minutes_double_counted'] / union_by_fleet['summed_minutes']).round(2)
print(union_by_fleet.sort_values('fleet').to_string(index=False))
print()


In [ ]:
# =============================================================================
# PS2 OVERLAP DIAGNOSTIC -- CELL 3 of 3: concurrency depth, and the restated metric
#
# C  How deep does concurrency go, and are the simultaneous episodes genuinely
#    different components (legitimate multi-component recording) or the same
#    thing recorded repeatedly (a data defect)? These need different fixes.
# E  Recompute the share of available device-hours from the union, clipped per
#    calendar day so no device can exceed 1440 minutes in a day by construction.
#
# READ-ONLY.
# =============================================================================
from pyspark.sql import functions as F, Window
from pyspark import StorageLevel
import pandas as pd

# ---------------------------------------------------------------- C: concurrency depth
# Sweep line per device: +1 at every start, -1 at every end. Ordering puts the
# -1 first at an identical timestamp so intervals that merely touch are not
# counted as concurrent.
print('C.1  How many OOS episodes are open at once on the same device?')
edges = (
    og.select('fleet', 'device_id', F.col('start_ts').alias('ts'), F.lit(1).alias('delta'))
    .unionByName(og.select('fleet', 'device_id', F.col('end_ts').alias('ts'), F.lit(-1).alias('delta')))
)
w_sweep = Window.partitionBy('fleet', 'device_id').orderBy('ts', 'delta').rowsBetween(Window.unboundedPreceding, Window.currentRow)
depth = edges.withColumn('open_now', F.sum('delta').over(w_sweep))
depth_by_device = depth.groupBy('fleet', 'device_id').agg(F.max('open_now').alias('max_concurrent'))
depth_summary = (
    depth_by_device.groupBy('fleet').agg(
        F.count('*').alias('devices'),
        F.sum(F.when(F.col('max_concurrent') <= 1, 1).otherwise(0)).alias('devices_never_concurrent'),
        F.expr('percentile_approx(max_concurrent, 0.5)').alias('p50_max_concurrent'),
        F.expr('percentile_approx(max_concurrent, 0.9)').alias('p90_max_concurrent'),
        F.max('max_concurrent').alias('worst_max_concurrent'),
    ).toPandas()
)
depth_summary['pct_devices_with_concurrency'] = (
    100.0 * (depth_summary['devices'] - depth_summary['devices_never_concurrent']) / depth_summary['devices']).round(2)
print(depth_summary.sort_values('fleet').to_string(index=False))
print()

# What distinguishes the episodes that share a merged block? If the subsystem or
# the serial differs, this is legitimate multi-component recording and the fix
# is to take the union. If they are identical, the rows are redundant and the
# fix is upstream de-duplication.
print('C.2  Within a merged block, are the concurrent episodes different components?')
blocks_multi = merged.where(F.col('episodes_in_block') > 1).select('fleet', 'device_id', '_block', 'block_start', 'block_end')
def _opt(name):
    # read_source projects an optional column only when the source has it, so
    # never assume. A missing column becomes a constant and the corresponding
    # distinct-count simply reads 1, which is visibly uninformative rather than
    # a crash or a silent wrong answer.
    return (F.coalesce(F.col(name).cast('string'), F.lit('(null)')) if name in og.columns
            else F.lit('(not projected)'))

tagged = og.select(
    'fleet', 'device_id', 'start_ts', 'end_ts',
    _opt('component_subsystem').alias('subsystem'),
    _opt('COMPONENT_SERIAL_NBR').alias('serial'),
    _opt('source_event_id').alias('event_id'),
)
print('optional columns available:',
      {c: (c in og.columns) for c in ('component_subsystem', 'COMPONENT_SERIAL_NBR', 'source_event_id')})
if True:
    joined = (
        tagged.alias('o').join(
            blocks_multi.alias('b'),
            (F.col('o.fleet') == F.col('b.fleet')) & (F.col('o.device_id') == F.col('b.device_id'))
            & (F.col('o.start_ts') >= F.col('b.block_start')) & (F.col('o.end_ts') <= F.col('b.block_end')),
            'inner')
        .select(F.col('b.fleet').alias('fleet'), F.col('b.device_id').alias('device_id'),
                F.col('b._block').alias('_block'), 'subsystem', 'serial', 'event_id')
    )
    composition = (
        joined.groupBy('fleet', 'device_id', '_block').agg(
            F.count('*').alias('episodes'),
            F.countDistinct('subsystem').alias('distinct_subsystems'),
            F.countDistinct('serial').alias('distinct_serials'),
            F.countDistinct('event_id').alias('distinct_event_ids'),
        )
        .groupBy('fleet').agg(
            F.count('*').alias('multi_episode_blocks'),
            F.sum(F.when(F.col('distinct_subsystems') > 1, 1).otherwise(0)).alias('blocks_multi_subsystem'),
            F.sum(F.when(F.col('distinct_serials') > 1, 1).otherwise(0)).alias('blocks_multi_serial'),
            F.sum(F.when(F.col('distinct_event_ids') < F.col('episodes'), 1).otherwise(0)).alias('blocks_with_repeated_event_id'),
            F.expr('percentile_approx(episodes, 0.5)').alias('p50_episodes'),
            F.expr('percentile_approx(distinct_subsystems, 0.5)').alias('p50_distinct_subsystems'),
        ).toPandas()
    )
    composition['pct_blocks_multi_subsystem'] = (
        100.0 * composition['blocks_multi_subsystem'] / composition['multi_episode_blocks'].clip(lower=1)).round(2)
    print(composition.sort_values('fleet').to_string(index=False))
print()

# ---------------------------------------------------------------- E: restated metric
# Split each merged block across calendar days and clip to the day. A device
# cannot exceed 1440 minutes in a day, so this is bounded by construction and
# any remaining implausibility is real.
print('E.1  Restated share of available device-hours, from the union, clipped per day')
exploded = (
    merged.select('fleet', 'device_id', 'block_start', 'block_end')
    .withColumn('day', F.explode(F.sequence(F.to_date('block_start'), F.to_date('block_end'))))
    .withColumn('day_start', F.col('day').cast('timestamp'))
    .withColumn('day_end', F.col('day_start') + F.expr('INTERVAL 1 DAY'))
    .withColumn('ov_start', F.greatest('block_start', 'day_start'))
    .withColumn('ov_end', F.least('block_end', 'day_end'))
    .withColumn('minutes_in_day',
                F.greatest(F.lit(0.0), (F.col('ov_end').cast('long') - F.col('ov_start').cast('long')) / 60.0))
)
device_day = (
    exploded.groupBy('fleet', 'device_id', 'day')
    .agg(F.sum('minutes_in_day').alias('oos_minutes'))
    .persist(StorageLevel.DISK_ONLY)
)
ceiling_breaches = device_day.where(F.col('oos_minutes') > 1440.5).count()
print(f'device-days exceeding 1440 minutes after the union: {ceiling_breaches:,}  (must be 0)')
print()

restated = (
    device_day.groupBy('fleet').agg(
        F.countDistinct('device_id').alias('devices'),
        F.countDistinct('day').alias('days_observed'),
        F.count('*').alias('device_days_with_oos'),
        F.sum('oos_minutes').alias('union_oos_minutes'),
        F.expr('percentile_approx(oos_minutes, 0.5)').alias('p50_minutes_per_affected_device_day'),
        F.expr('percentile_approx(oos_minutes, 0.9)').alias('p90_minutes_per_affected_device_day'),
    ).toPandas()
)
old = union_by_fleet[['fleet', 'summed_minutes', 'union_minutes']]
restated = restated.merge(old, on='fleet')
restated['capacity_device_hours'] = (restated['devices'] * WIN_DAYS * DAY / 60.0).round(1)
restated['old_share_pct'] = (100.0 * restated['summed_minutes'] / (restated['devices'] * WIN_DAYS * DAY)).round(2)
restated['restated_share_pct'] = (100.0 * restated['union_oos_minutes'] / (restated['devices'] * WIN_DAYS * DAY)).round(2)
restated['overstatement_factor'] = (restated['old_share_pct'] / restated['restated_share_pct'].clip(lower=0.001)).round(2)
restated['pct_device_days_affected'] = (
    100.0 * restated['device_days_with_oos'] / (restated['devices'] * WIN_DAYS)).round(2)
cols = ['fleet', 'devices', 'capacity_device_hours', 'old_share_pct', 'restated_share_pct',
        'overstatement_factor', 'pct_device_days_affected',
        'p50_minutes_per_affected_device_day', 'p90_minutes_per_affected_device_day']
print(restated[cols].sort_values('fleet').to_string(index=False))
print()

print('=' * 78)
print('HOW TO READ THIS')
print('=' * 78)
print(
    'A.1 decides what duration_min even is. If it agrees with (end - start), the sum\n'
    '    and the union are comparable. If duration_min is systematically smaller, it is\n'
    '    already a clipped quantity and the union is the only honest measure.\n'
    'B.1 is the decisive line. overlap_factor is how many times the average out-of-service\n'
    '    minute is counted. 1.0 means no overlap and the current metric is arithmetically\n'
    '    sound. Well above 1.0 means PS2 is summing concurrent episodes and the published\n'
    '    share is inflated by exactly that factor.\n'
    'C.1 says whether concurrency is rare or the norm. C.2 says what KIND it is:\n'
    '    blocks_multi_subsystem high  -> legitimate multi-component recording; the fix is\n'
    '                                    the union, in the PS2 notebook, and nothing else\n'
    '    blocks_with_repeated_event_id high -> the same episode is present more than once;\n'
    '                                    that is a source or ingestion defect and belongs\n'
    '                                    in a conversation with Cubic, not a notebook patch\n'
    'E.1 is what the dashboard should say. The ceiling check must print 0: after taking the\n'
    '    union no device can be out of service for more than 1440 minutes in a day. If it\n'
    '    prints 0 and restated_share_pct is plausible, the metric is fixed and the change\n'
    '    is one aggregation in the PS2 notebook.\n'
)
print(
    'WHAT THIS STILL DOES NOT TEST. Whether the underlying episodes are themselves real.\n'
    'A union of spurious intervals is still spurious. It also says nothing about data\n'
    'after 11 April 2026; the closure diagnostic already showed boundary censoring is\n'
    'negligible here, at a few thousand rows out of millions.\n'
)
